# **CV #2: Object Detection**

by **R. Ethan Halim** (CS-A 2024, 536718)

⚠️ **The folder of this assignment MUST BE downloaded and viewed locally, in order to be able to show any videos in the notebook, as it unfortunately cannot be displayed through GitHub.**

## Auxiliary Preface

This assignment primarily uses [numpy](https://numpy.org) for array computations, [numba](https://numba.pydata.org) for performance acceleration, [pygame-ce](https://pyga.me) for text and drawing, and [OpenCV](https://opencv.org) for loading videos. None of these libraries implement the CV algorithms used in this assignment. Again, OpenCV is **solely** used for video loading.

Below are the imports as well as a couple of auxiliary functions.

In [142]:
import numpy as np
import pygame as pg
import cv2

from IPython.display import display, Video
from numba import jit, njit, prange


pg.init()
box_font = pg.font.SysFont(None, 32)
caption_font = pg.font.SysFont(None, 48)
fps: int


## Helper function to load video files and display in the notebook
def load_video(filename: str) -> list[np.ndarray]:
    global fps
    cap = cv2.VideoCapture(filename)
    assert cap.isOpened()

    fps = int(cap.get(cv2.CAP_PROP_FPS))
    frames: list[np.ndarray] = []
    while True:
        success, frame_bgr = cap.read()
        if not success:
            break

        frame_rgb: np.ndarray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        frames.append(frame_rgb.astype(np.float32) / 255.0)

    cap.release()
    display(Video(filename, height=480))
    return frames


## Helper function to save video files and display in the notebook
def save_video(frames: list[np.ndarray], filename: str):
    height, width, _ = frames[0].shape
    fourcc = cv2.VideoWriter_fourcc(*"avc1")
    out = cv2.VideoWriter(filename, fourcc, fps, (width, height))
    assert out.isOpened()

    for frame_rgb in frames:
        frame_rgb = (frame_rgb * 255.0).astype(np.uint8)
        out.write(cv2.cvtColor(frame_rgb, cv2.COLOR_RGB2BGR))

    out.release()
    display(Video(filename, height=480))


## Helper function to draw boxes and text
def draw(frame: np.ndarray, boxes: list[tuple[pg.Rect, str, str]], caption: str) -> np.ndarray:
    surf = pg.surfarray.make_surface((frame.transpose(1, 0, 2) * 255.0).astype(np.uint8))

    for rect, text, color in boxes:
        pg.draw.rect(surf, color, rect, 4)
        surf.blit(
            box_font.render(text, True, "black"),
            (rect.x + 10, rect.y + 10)
        )
        surf.blit(
            box_font.render(text, True, color),
            (rect.x + 8, rect.y + 8)
        )

    surf.blit(caption_font.render(caption, True, "black"), (18, 18))
    surf.blit(caption_font.render(caption, True, "white"), (16, 16))

    return pg.surfarray.pixels3d(surf).transpose(1, 0, 2).astype(np.float32) / 255.0

## Materials

### Footage

Below, I acquired an altered [stock video](https://motionarray.com/stock-video/man-walking-111417) of two men walking across a pavement. The goal is to be able to detect and locate their positions as they walk across the frame.

In [143]:
video = load_video("video.mp4")

### Annotations

In [144]:
import json # Built-in module

annotations_json = json.load(open(r"annotations.coco.json"))

video_image_ids: dict[int, int] = {}
for image in annotations_json["images"]:
    name = image["file_name"]
    if name.startswith("video-") and name.endswith(".jpg"):
        video_image_ids[image["id"]] = int(name[6:-4]) - 1

annotations: list[list[pg.Rect]] = [[] for _ in video]
for annotation in annotations_json["annotations"]:
    x, y, w, h = map(lambda v: int(float(v)), annotation["bbox"])
    annotations[video_image_ids[annotation["image_id"]]].append(
        pg.Rect((x, y), (w, h))
    )

In [145]:
detection_frames: list[np.ndarray] = []
for mask, annotation in zip(video, annotations):
    detection_frames.append(
        draw(
            mask,
            [(rect, "", "yellow") for rect in annotation],
            f"{len(annotation)} annotation(s)"
        )
    )

save_video(detection_frames, "generated/annotated.mp4")

## Background Modeling

### Mean Image

In [146]:
def get_image_mean(frames: list[np.ndarray]) -> np.ndarray:
    return sum(frames) / len(frames) # type: ignore

In [147]:
mean_frames: list[np.ndarray] = []
for i in range(len(video)):
    mean_frames.append(get_image_mean(video[:i+1]))

save_video(mean_frames, "generated/mean.mp4")


In [148]:
get_image_mean = mean_frames[-1]
del mean_frames

### Thresholding through Background Subtraction

In [149]:
def threshold_subtraction(frame: np.ndarray, background: np.ndarray, threshold: float) -> np.ndarray:
    difference = np.abs(frame - background)
    return np.all(difference >= threshold, 2)[..., None]

In [150]:
masks: list[np.ndarray] = []
mask_frames: list[np.ndarray] = []

for mask in video:
    mask = threshold_subtraction(mask, get_image_mean, 0.2)
    masks.append(mask)
    mask_frames.append(
        np.where(
            mask,
            np.array((1.0, 1.0, 1.0), dtype=np.float32),
            np.array((0.0, 0.0, 0.0), dtype=np.float32)
        )
    )

save_video(mask_frames, "generated/mask.mp4")
del mask_frames

### Mode (Median) Filter

In [151]:
@njit
def filter_mode(mask: np.ndarray, radius: int) -> np.ndarray:
    filtered = np.empty(mask.shape, np.bool)
    kernel_size = radius * 2 + 1
    kernel_area = kernel_size * kernel_size

    for y in prange(mask.shape[0]):
        for x in prange(mask.shape[1]):
            ones = 0
            for dy in range(y - radius, y + radius + 1):
                if not (0 <= dy < mask.shape[0]):
                    continue
                for dx in range(x - radius, x + radius + 1):
                    if 0 <= dx < mask.shape[1] and mask[dy][dx][0]:
                        ones += 1

            if ones > kernel_area // 2:
                filtered[y][x][0] = True
            else:
                filtered[y][x][0] = False

    return filtered

In [152]:
filtered_masks: list[np.ndarray] = []
filtered_mask_frames: list[np.ndarray] = []

for mask in masks:
    filtered_mask = filter_mode(mask, 2)
    filtered_masks.append(filtered_mask)
    filtered_mask_frames.append(
        np.where(
            filtered_mask,
            np.array((1.0, 1.0, 1.0), dtype=np.float32),
            np.array((0.0, 0.0, 0.0), dtype=np.float32)
        )
    )

save_video(filtered_mask_frames, "generated/filtered.mp4")
del filtered_mask_frames

### Foreground Separation

In [153]:
fg_frames: list[np.ndarray] = []
for frame, mask in zip(video, filtered_masks):
    fg_frames.append(frame * mask)

save_video(fg_frames, "generated/foreground.mp4")

## Connected-Component Labeling

In [154]:
@njit
def set_find(parents: np.ndarray, x: int):
    while parents[x] != x:
        # Path compression
        parents[x] = parents[parents[x]]
        x = parents[x]

    return x


@njit
def set_union(parents: np.ndarray, a: int, b: int):
    ra = set_find(parents, a)
    rb = set_find(parents, b)

    if ra != rb:
        if ra < rb:
            parents[rb] = ra
        else:
            parents[ra] = rb

In [155]:
@njit
def label_connected(mask: np.ndarray) -> np.ndarray:
    h, w, _ = mask.shape
    out = np.zeros((h, w, 1), dtype=np.uint32)

    # Worst case: every other pixel is its own component
    parents = np.empty(h * w + 1, dtype=np.uint32)
    parents[0] = 0

    next_label = 1

    # First pass: assign provisional labels + record equivalences
    for i in range(h):
        for j in range(w):
            if not mask[i, j, 0]:
                continue

            up = out[i - 1, j, 0] if i > 0 else 0
            left = out[i, j - 1, 0] if j > 0 else 0

            if up == 0 and left == 0:
                out[i, j, 0] = next_label
                parents[next_label] = next_label
                next_label += 1
            elif up != 0 and left == 0:
                out[i, j, 0] = up
            elif up == 0 and left != 0:
                out[i, j, 0] = left
            else:
                out[i, j, 0] = up
                set_union(parents, up, left)

    # Second pass: flatten labels and remap to compact IDs 1..N
    root_to_new = np.zeros(next_label, dtype=np.uint32)
    new_id = 1

    for i in range(h):
        for j in range(w):
            lab = out[i, j, 0]
            if lab == 0:
                continue

            root = set_find(parents, lab)
            if root_to_new[root] == 0:
                root_to_new[root] = new_id
                new_id += 1
            out[i, j, 0] = root_to_new[root]

    return out

In [156]:
components: list[np.ndarray] = []
component_frames: list[np.ndarray] = []

for mask in filtered_masks:
    component_frame = label_connected(mask)
    components.append(component_frame)
    component_frames.append(
        np.ones((*component_frame.shape[:2], 3), np.float32)
        * component_frame
        / max(1, np.max(component_frame))
    )

save_video(component_frames, "generated/connected.mp4")
del component_frames

## Bounding Boxes

### Bounding

In [157]:
@njit
def _bound_boxes(component_areas: np.ndarray, component_bounds: np.ndarray, components: np.ndarray):
    for y in range(components.shape[0]):
        for x in range(components.shape[1]):
            component = components[y][x][0]
            if component == 0:
                continue

            component -= 1
            component_areas[component] += 1
            
            ax, ay, bx, by = component_bounds[component]
            ax = x if x < ax else ax
            ay = y if y < ay else ay
            bx = x if x > bx else bx
            by = y if y > by else by
            component_bounds[component] = np.array((ax, ay, bx, by), np.uint32)


def bound_boxes(components: np.ndarray, min_area: int) -> list[pg.Rect]:
    component_count = np.max(components)
    component_areas = np.zeros((component_count,), np.uint32)
    component_bounds = np.tile(
        np.array((components.shape[1], components.shape[0], 0, 0), np.uint32),
        (component_count, 1)
    )

    _bound_boxes(component_areas, component_bounds, components)

    boxes: list[pg.Rect] = []
    for component in range(component_count):
        if component_areas[component] >= min_area:
            ax, ay, bx, by = component_bounds[component]
            boxes.append(pg.Rect((ax, ay, bx - ax, by - ay)))

    return boxes

In [158]:
bboxes = [bound_boxes(component, 10_000) for component in components]
bbox_frames: list[np.ndarray] = []

for frame, bbox in zip(video, bboxes):
    bbox_frames.append(
        draw(
            frame,
            [(rect, "", "yellow") for rect in bbox],
            f"{len(bbox)} box(es)"
        )
    )

save_video(bbox_frames, "generated/bbox.mp4")
del bbox_frames

### Merging Overlaps

In [159]:
def merge_overlaps(bboxes: list[pg.Rect]) -> list[pg.Rect]:
    for i, bbox in enumerate(bboxes):
        if bbox is None:
            continue

        for j in range(i + 1, len(bboxes)):
            other_bbox = bboxes[j]
            if other_bbox is not None and bbox.colliderect(bboxes[j]):
                bbox = bbox.union(bboxes[j])
                bboxes[j] = None # type: ignore
        
        bboxes[i] = bbox
    
    return list(filter(lambda x: x is not None, bboxes))

In [160]:
detections: list[list[pg.Rect]] = []
detection_frames: list[np.ndarray] = []

for frame, bbox in zip(video, bboxes):
    detection = merge_overlaps(bbox)
    detections.append(detection)
    detection_frames.append(
        draw(
            frame,
            [(rect, "", "yellow") for rect in detection],
            f"{len(detection)} detection(s)"
        )
    )

save_video(detection_frames, "generated/detected.mp4")
del detection_frames

## Scoring

### Intersection over Union

In [161]:
def get_iou(bbox: pg.Rect, reference_bboxes: list[pg.Rect]) -> tuple[pg.Rect | None, float]:
    area = bbox.width * bbox.height

    max_iou = 0.0
    rect = None
    for reference in reference_bboxes:
        if not bbox.colliderect(reference):
            continue

        clipping = bbox.clip(reference)
        intersection = clipping.width * clipping.height
        union = area + reference.width * reference.height - intersection

        iou = intersection / union
        if iou > max_iou:
            rect = reference
            max_iou = iou

        max_iou = max(max_iou, intersection / union)

    return rect, max_iou

### Accuracy

In [164]:
threshold = 0.50
total_tp = total_fp = total_fn = total_annotations = 0

scoring_frames: list[np.ndarray] = []
for frame, detection, annotation in zip(video, detections, annotations):
    detected_annotations: list[pg.Rect] = []
    true_detections: dict[tuple[int, int, int, int], float] = {}
    false_detections: dict[tuple[int, int, int, int], float] = {}

    for bbox in detection:
        target, iou = get_iou(bbox, annotation)
        if target is not None and iou >= threshold:
            detected_annotations.append(target)
            true_detections[(bbox.x, bbox.y, bbox.w, bbox.h)] = iou
        else:
            false_detections[(bbox.x, bbox.y, bbox.w, bbox.h)] = iou

    total_tp += (tp := len(true_detections))
    total_fp += (fp := len(false_detections))
    total_fn += (fn := len(annotation) - len(detected_annotations))
    total_annotations += len(annotation)

    accuracy = tp / len(annotation) if len(annotation) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    scoring_frames.append(
        draw(
            frame,
            [(rect, "", "gray") for rect in annotation]
            + [(rect, "", "yellow") for rect in detected_annotations]
            + [(pg.Rect(rect), f"{iou * 100:.2f}%", "green") for rect, iou in true_detections.items()]
            + [(pg.Rect(rect), f"{iou * 100:.2f}%", "red") for rect, iou in false_detections.items()],
            f"TP: {tp}   FP: {fp}   FN: {fn}\n"
            f"Accuracy: {accuracy:.4f}\n"
            f"Precision: {precision:.4f}\n"
            f"Recall: {recall:.4f}"
        )
    )

save_video(scoring_frames, "generated/scoring.mp4")
del scoring_frames

In [165]:
print(
    f"Accuracy\t: {total_tp / total_annotations * 100:.4f}%\n"
    f"Precision\t: {total_tp / (total_tp + total_fp) * 100:.4f}%\n"
    f"Recall\t\t: {total_tp / (total_tp + total_fn) * 100:.4f}%"
)

Accuracy	: 62.1469%
Precision	: 61.1111%
Recall		: 62.1469%
